In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

def selenium_scrape_image_url():
    # Selenium 옵션 설정 (백그라운드 실행을 원할 경우)
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # 크롬 드라이버 설정
    service = Service(executable_path='../lib/chromedriver-win64/chromedriver.exe')  # chromedriver 경로 설정
    driver = webdriver.Chrome(service=service, options=chrome_options)

    # URL 접속
    driver.get("https://n.news.naver.com/article/053/0000046132?cds=news_media_pc&type=editn")

    # img 태그와 src 속성 추출
    img_element = driver.find_element(By.ID, "img1")
    img_url = img_element.get_attribute("src")
    
    print(f"Image URL: {img_url}")
    driver.quit()
    return img_url

# 실행
selenium_scrape_image_url()

Image URL: https://imgnews.pstatic.net/image/053/2024/10/06/0000046132_001_20241006154608161.jpg?type=w860


'https://imgnews.pstatic.net/image/053/2024/10/06/0000046132_001_20241006154608161.jpg?type=w860'

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from concurrent.futures import ThreadPoolExecutor
import pandas as pd


class SeleniumDriver:
    def __init__(self):
        self.display = None
        self.driver = None
    
    def setup(self, options=None):
        caps = DesiredCapabilities.CHROME
        caps["pageLoadStrategy"] = "none"

        chrome_options = webdriver.ChromeOptions()

        if options:
            for opt in options:
                chrome_options.add_argument(opt)
        
        if DISPLAY_AVAILABLE:
            self.display = Display(visible=0, size=(1920, 1080))
            self.display.start()

        self.driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options, desired_capabilities=caps)
    
    def teardown(self):
        if self.display:
            self.display.stop()
        if self.driver:
            self.driver.quit()

def process_id(id, driver):
    driver.get(f"https://www.shop.com/{id}")

    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".product_title"))
    )

    product_details = extract_product_details(id, driver)

    return product_details

def main():
    id_list = [...]

    driver = SeleniumDriver()
    driver.setup()
    
    results = []

    with ThreadPoolExecutor() as executor:
        futures = []

        for id in id_list:
            futures.append(executor.submit(process_id, id, driver.driver))

        for future in concurrent.futures.as_completed(futures):
            results.append(future.result())

    driver.teardown()

    product_details_df = pd.DataFrame([r["product_details"] for r in results])

if __name__ == '__main__':
    main()

NameError: name 'DISPLAY_AVAILABLE' is not defined

In [18]:
import os
import requests
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# 네이버 API 키 가져오기
client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECRET')

def naver_search(query, display=10, start=1):
    # 네이버 검색 API 요청 URL
    url = "https://openapi.naver.com/v1/search/news.json"
    
    # 요청 헤더 설정
    headers = {
        "X-Naver-Client-Id": client_id,
        "X-Naver-Client-Secret": client_secret
    }
    
    # 요청 파라미터 설정
    params = {
        "query": query,
        "display": display,  # 한 번에 표시할 검색 결과 개수
        "start": start       # 검색 시작 위치
    }
    
    # API 요청 보내기
    response = requests.get(url, headers=headers, params=params)
    
    # 상태 코드 200일 때만 데이터 반환
    if response.status_code == 200:
        # 응답 데이터에서 네이버 뉴스 링크만 추출
        data = response.json()
        naver_links = [item['link'] for item in data['items'] if 'n.news.naver.com' in item['link']]
        return naver_links
    else:
        print(f"Error Code: {response.status_code}")
        return None


# 예시: '환율'에 대한 뉴스 검색
search_result = naver_search('환율')

# 검색 결과 출력 (네이버 뉴스 링크만 출력)
if search_result:
    for link in search_result:
        print(f"Link: {link}")
else:
    print("검색 결과가 없습니다.")


Link: https://n.news.naver.com/mnews/article/050/0000080767?sid=101
Link: https://n.news.naver.com/mnews/article/448/0000482459?sid=101
Link: https://n.news.naver.com/mnews/article/214/0001379597?sid=101
Link: https://n.news.naver.com/mnews/article/088/0000909160?sid=101


In [1]:
import csv

# 데이터 정의
competitors = [
    "삼성SDS", "LG CNS", "현대오토에버", "SK C&C", "롯데정보통신",
    "포스코DX", "미라콤아이앤씨", "메가존클라우드", "한화시스템", "CJ올리브네트웍스"
]

it_trends = [
    "생성형 AI", "머신러닝", "딥러닝", "자연어 처리", "컴퓨터 비전",
    "로보틱스", "자율주행", "메타버스", "가상현실", "증강현실",
    "혼합현실", "블록체인", "암호화폐", "NFT", "클라우드 컴퓨팅",
    "엣지 컴퓨팅", "5G/6G 네트워크", "IOT", "빅데이터", "데이터 애널리틱스",
    "사이버보안", "양자 컴퓨팅", "디지털 트윈", "3D 프린팅", "드론 기술",
    "웨어러블 기술", "스마트 홈", "스마트 시티", "그린 IT", "지속가능한 기술",
    "로우코드/노코드 개발", "DevOps", "MLOps", "마이크로서비스 아키텍처", "컨테이너화",
    "서버리스 컴퓨팅", "API 경제", "디지털 변환", "핀테크", "헬스테크"
]

# CSV 파일 생성
with open('it_companies_and_trends.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    
    # 헤더 작성
    writer.writerow(['Competitors', 'IT Trends'])
    
    # 데이터 작성
    max_length = max(len(competitors), len(it_trends))
    for i in range(max_length):
        competitor = competitors[i] if i < len(competitors) else ''
        trend = it_trends[i] if i < len(it_trends) else ''
        writer.writerow([competitor, trend])

print("CSV 파일이 성공적으로 생성되었습니다.")

CSV 파일이 성공적으로 생성되었습니다.


In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import pandas as pd

# Semantic Splitter Function
def semantic_split(text, threshold=0.7):
    sentences = text.split('. ')  # 문장 단위로 분리
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(sentences)
    
    chunks = []
    current_chunk = []
    for i, sentence in enumerate(sentences):
        if i > 0 and cosine_similarity([embeddings[i]], [embeddings[i-1]]) < threshold:
            chunks.append(' '.join(current_chunk))
            current_chunk = []
        current_chunk.append(sentence)
    chunks.append(' '.join(current_chunk))
    return chunks

# Word Frequency Function
def calculate_word_frequencies(chunks, stopwords=None):
    word_counter = Counter()
    for chunk in chunks:
        words = chunk.split()
        filtered_words = [word for word in words if not stopwords or word not in stopwords]
        word_counter.update(filtered_words)
    return word_counter

# 한글 불용어 리스트 정의
STOPWORDS = set([
    "그리고", "하지만", "수", "있다", "그", "이", "것", "때문에", "또는", "등", "및", "대한", "으로", "에서", 
    "의", "이다", "가", "을", "에", "와", "과", "도", "는", "다", "한", "하고", "으로서", "한다", "이다", "있으며", 
    "하고", "하게", "였다", "라고", "라고도", "하면서"
])

# 긴 텍스트 입력
large_text = """고위공직자범죄수사처가 윤석열 대통령에게 3차례에 걸쳐 출석을 요구했지만 윤 대통령은 출석요구서 수령을 거절하고 출석과 관련해 어떠한 연락도 하지 않는 '무대응' 전략으로 맞서고 있다.

윤 대통령이 지난 18일과 25일에 조사받으라는 1·2차 출석 요구에 불응한 이후 공수처는 윤 대통령에게 29일 오전 10시 정부과천청사로 나와 내란 우두머리 및 직권남용 혐의 피의자 신분으로 조사받으라는 3차 출석요구서를 보냈다.

통상 수사기관은 피의자에 대해 세 차례 출석 요구를 한 뒤에도 타당한 이유 없이 조사에 응하지 않으면 법원에 체포영장을 청구한다.

3차 출석요구가 사실상 최후통첩인 셈이지만, 윤 대통령 측은 이번에도 우편물 수령을 거부한 것으로 파악됐다.

윤 대통령 측 석동현 변호사는 지난 24일 '수사보다 탄핵 심판이 우선'이라며 당분간 조사에 응할 계획이 없음을 시사했다.

윤 대통령은 이번 계엄 사태의 정점에서 내란 중요임무 종사 혐의를 받는 이들에게 지시를 내린 것으로 지금까지 검찰 수사에서 나타났다.

검찰은 김용현 전 국방부 장관을 재판에 넘기면서 공소장을 통해 윤 대통령의 지시 사항을 구체적으로 적시했다.

윤 대통령이 3월부터 비상계엄을 염두에 뒀고, 실질적 준비는 지난달부터 진행됐다고 한 것이다.

또 사령관들에게 여야 대표 신병 확보·국회의원 체포·계엄 해제 요구안 의결 저지 등을 지시한 최종 '윗선'이 윤 대통령이라고 했다.

하지만 윤 대통령에 대한 직접 조사는 아직 이뤄지지 않았다. 공수처의 이첩 요청에 따라 사건이 넘어가 검찰이 윤 대통령을 언제 조사할 수 있을지도 미지수다.

공수처법상 검경은 수사 진행 정도·공정성 논란 등에 비춰 공수처가 적절하다고 판단해 중복 사건의 이첩을 요청하면 응해야 한다.

윤 대통령에 다한 공수처 수사가 '장기전'으로 진행되는 가운데 검찰이 윤 대통령을 직접 부르는 건 공수처 수사 이후에나 가능하다.

공수처는 대통령에 대해선 법률상 기소권이 없어 기소하려면 검찰을 거쳐야 한다. 공수처가 기소를 위해 사건 자료 등을 검찰에 송부하면, 검찰이 보완수사에 나설 수 있다."""

# Semantic Split
chunks = semantic_split(large_text)

# Calculate Word Frequencies
word_frequencies = calculate_word_frequencies(chunks, stopwords=STOPWORDS)

# Convert to DataFrame for Sorting and Display
word_freq_df = pd.DataFrame(word_frequencies.items(), columns=['Word', 'Count']).sort_values(by='Count', ascending=False)

# Display Results
print("=== Semantic Chunks ===")
for i, chunk in enumerate(chunks):
    print(f"[Chunk {i + 1}]: {chunk}\n")

print("=== Word Frequency ===")
print(word_freq_df)


=== Semantic Chunks ===
[Chunk 1]: 고위공직자범죄수사처가 윤석열 대통령에게 3차례에 걸쳐 출석을 요구했지만 윤 대통령은 출석요구서 수령을 거절하고 출석과 관련해 어떠한 연락도 하지 않는 '무대응' 전략으로 맞서고 있다.

윤 대통령이 지난 18일과 25일에 조사받으라는 1·2차 출석 요구에 불응한 이후 공수처는 윤 대통령에게 29일 오전 10시 정부과천청사로 나와 내란 우두머리 및 직권남용 혐의 피의자 신분으로 조사받으라는 3차 출석요구서를 보냈다.

통상 수사기관은 피의자에 대해 세 차례 출석 요구를 한 뒤에도 타당한 이유 없이 조사에 응하지 않으면 법원에 체포영장을 청구한다.

3차 출석요구가 사실상 최후통첩인 셈이지만, 윤 대통령 측은 이번에도 우편물 수령을 거부한 것으로 파악됐다.

윤 대통령 측 석동현 변호사는 지난 24일 '수사보다 탄핵 심판이 우선'이라며 당분간 조사에 응할 계획이 없음을 시사했다.

윤 대통령은 이번 계엄 사태의 정점에서 내란 중요임무 종사 혐의를 받는 이들에게 지시를 내린 것으로 지금까지 검찰 수사에서 나타났다.

검찰은 김용현 전 국방부 장관을 재판에 넘기면서 공소장을 통해 윤 대통령의 지시 사항을 구체적으로 적시했다.

윤 대통령이 3월부터 비상계엄을 염두에 뒀고, 실질적 준비는 지난달부터 진행됐다고 한 것이다.

또 사령관들에게 여야 대표 신병 확보·국회의원 체포·계엄 해제 요구안 의결 저지 등을 지시한 최종 '윗선'이 윤 대통령이라고 했다.

하지만 윤 대통령에 대한 직접 조사는 아직 이뤄지지 않았다 공수처의 이첩 요청에 따라 사건이 넘어가 검찰이 윤 대통령을 언제 조사할 수 있을지도 미지수다.

공수처법상 검경은 수사 진행 정도·공정성 논란 등에 비춰 공수처가 적절하다고 판단해 중복 사건의 이첩을 요청하면 응해야 한다.

윤 대통령에 다한 공수처 수사가 '장기전'으로 진행되는 가운데 검찰이 윤 대통령을 직접 부르는 건 공수처 수사 이후에나 가능하다.

공수처는 대통령에 대해선 법률상 기소권이 

In [8]:
import kss
from sentence_transformers import SentenceTransformer

# 텍스트와 모델
text = """고위공직자범죄수사처가 윤석열 대통령에게 3차례에 걸쳐 출석을 요구했지만 윤 대통령은 출석요구서 수령을 거절하고 출석과 관련해 어떠한 연락도 하지 않는 '무대응' 전략으로 맞서고 있다.

윤 대통령이 지난 18일과 25일에 조사받으라는 1·2차 출석 요구에 불응한 이후 공수처는 윤 대통령에게 29일 오전 10시 정부과천청사로 나와 내란 우두머리 및 직권남용 혐의 피의자 신분으로 조사받으라는 3차 출석요구서를 보냈다.

통상 수사기관은 피의자에 대해 세 차례 출석 요구를 한 뒤에도 타당한 이유 없이 조사에 응하지 않으면 법원에 체포영장을 청구한다.

3차 출석요구가 사실상 최후통첩인 셈이지만, 윤 대통령 측은 이번에도 우편물 수령을 거부한 것으로 파악됐다.

윤 대통령 측 석동현 변호사는 지난 24일 '수사보다 탄핵 심판이 우선'이라며 당분간 조사에 응할 계획이 없음을 시사했다.

윤 대통령은 이번 계엄 사태의 정점에서 내란 중요임무 종사 혐의를 받는 이들에게 지시를 내린 것으로 지금까지 검찰 수사에서 나타났다.

검찰은 김용현 전 국방부 장관을 재판에 넘기면서 공소장을 통해 윤 대통령의 지시 사항을 구체적으로 적시했다.

윤 대통령이 3월부터 비상계엄을 염두에 뒀고, 실질적 준비는 지난달부터 진행됐다고 한 것이다.

또 사령관들에게 여야 대표 신병 확보·국회의원 체포·계엄 해제 요구안 의결 저지 등을 지시한 최종 '윗선'이 윤 대통령이라고 했다.

하지만 윤 대통령에 대한 직접 조사는 아직 이뤄지지 않았다. 공수처의 이첩 요청에 따라 사건이 넘어가 검찰이 윤 대통령을 언제 조사할 수 있을지도 미지수다.

공수처법상 검경은 수사 진행 정도·공정성 논란 등에 비춰 공수처가 적절하다고 판단해 중복 사건의 이첩을 요청하면 응해야 한다.

윤 대통령에 다한 공수처 수사가 '장기전'으로 진행되는 가운데 검찰이 윤 대통령을 직접 부르는 건 공수처 수사 이후에나 가능하다.

공수처는 대통령에 대해선 법률상 기소권이 없어 기소하려면 검찰을 거쳐야 한다. 공수처가 기소를 위해 사건 자료 등을 검찰에 송부하면, 검찰이 보완수사에 나설 수 있다."""
model = SentenceTransformer('snunlp/KR-SBERT-V1')

# 문장 분리
sentences = kss.split_sentences(text)

# 문장 임베딩
embeddings = model.encode(sentences)

# 출력
for i, sentence in enumerate(sentences):
    print(f"[문장 {i + 1}]: {sentence}")
    print(f"임베딩: {embeddings[i]}")

[Kss]: No sentence-transformers model found with name snunlp/KR-SBERT-V1. Creating a new one with mean pooling.


OSError: snunlp/KR-SBERT-V1 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [6]:
import pandas as pd
import os
import sqlite3

# 데이터 경로 설정
data_path = '../data/raw/reddit_data.csv'
text_save_dir = '../data/texts/'
db_path = '../data/processed/reddit_metadata.db'

# 디렉터리가 존재하지 않으면 생성
os.makedirs(text_save_dir, exist_ok=True)

# 데이터 불러오기
df = pd.read_csv(data_path)

# 텍스트 데이터 파일로 저장 및 경로 정보 수집
metadata = []
for idx, row in df.iterrows():
    # 텍스트 내용 저장
    text_content = row['Content']
    file_name = f'reddit_{idx}.txt'
    file_path = os.path.join(text_save_dir, file_name)
    
    # 텍스트 파일로 저장
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(text_content)
    
    # 메타데이터에 파일 경로 추가
    metadata.append({
        "Title": row['Title'],
        "Date": row['Date'],
        "Upvotes": row['Upvotes'],
        "Comments": row['Comments'],
        "Post_URL": row['Post URL'],
        "Content_URL": row['Content URL'],
        "File_Path": file_path  # 텍스트 파일 경로
    })

# 메타데이터를 DataFrame으로 변환
metadata_df = pd.DataFrame(metadata)

# SQLite 데이터베이스에 저장
conn = sqlite3.connect(db_path)
metadata_df.to_sql('reddit', conn, if_exists='replace', index=False)
conn.close()

print(f"Metadata saved to database at {db_path}")
print(f"Text contents saved in directory: {text_save_dir}")


Metadata saved to database at ../data/processed/reddit_metadata.db
Text contents saved in directory: ../data/texts/


In [13]:
import pandas as pd

# 입력한 문자열을 리스트 형태로 구성
input_text = """
삼성 SDS
LG CNS
현대오토에버
SK C&C
메가존클라우드
포스코DX
롯데정보통신
CJ올리브네트웍스
한화시스템
쌍용정보통신
클라우드 컴퓨팅
빅데이터
인공지능
데이터 분석
사물인터넷
엣지 컴퓨팅
디지털 트윈
사이버 보안
블록체인
소프트웨어 정의 네트워크
ERP
RPA
스마트 팩토리
스마트 시티
하이퍼오토메이션
IT 아웃소싱
클라우드 마이그레이션
데이터센터
AI 챗봇
네트워크 인프라
모바일 애플리케이션
컨테이너
마이크로서비스
소프트웨어 아키텍처
서버리스 컴퓨팅
API
스마트 홈
정보 보호
비즈니스 인텔리전스
IT 서비스 관리
데이터 웨어하우스
IT 거버넌스
정보 보안 관리
클라우드 네이티브
하이브리드 클라우드
데이터 레이크
전산화 업무
서버 가상화
네트워크 보안
기술 혁신
"""

# 문자열을 줄 단위로 분리하여 리스트로 변환
keywords = input_text.strip().split('\n')

# DataFrame 생성
df = pd.DataFrame(keywords, columns=["키워드"])

# CSV 파일로 저장
df.to_csv("keywords.csv", index=False, encoding='utf-8-sig')
print("CSV 파일이 성공적으로 저장되었습니다: keywords.csv")


CSV 파일이 성공적으로 저장되었습니다: keywords.csv
